In [1]:
"""
mv_RL.py — Reinforcement Learning for Jump-Diffusions
Mean-Variance Portfolio Selection — Off-Policy q-Learning
==========================================================
Trains the algorithm on three simulators and saves
results to mv_results.npz for mv_plot.py.

Simulatori:
  (A) Dati reali S&P 500 (bootstrap resampling)
  (B) Simulazione sintetica BS calibrata
  (C) Simulazione sintetica MJD calibrata

Output:
  mv_results.npz  — risultati da passare a mv_plot.py

Riferimenti:
  Gao, Li, Zhou (2024) arXiv:2405.16449
  Huang, Jia, Zhou (2022) ACM ICAIF
"""

import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm, poisson

np.random.seed(42)

# ─────────────────────────────────────────────────────────────────────────────
# 1.  PARAMETRI GLOBALI
# ─────────────────────────────────────────────────────────────────────────────

R_F         = 0.02
DT          = 1 / 12
T           = 1.0
K           = int(T / DT)
THETA       = 0.1
Z           = 1.4
X0          = 1.0

N_ITER      = 20000
BATCH       = 32
DECAY       = 0.51
ALPHA_PSI   = 0.001
ALPHA_OMEGA = 0.05
OMEGA_FREQ  = 10
GRAD_CLIP   = 5.0

# Adam per phi — coerente con mv_plot.py e mv_analysis.py
ADAM_LR    = 0.001
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.999
ADAM_EPS   = 1e-8


# ─────────────────────────────────────────────────────────────────────────────
# 2.  DOWNLOAD E PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

def load_sp500(start='2000-01-01', end='2023-12-31'):
    """
    Download full S&P 500 history 2000-2023 and return:
      - log_ret_daily:    daily log returns (full period)
      - log_ret_monthly:  monthly log returns (full period)
      - ret_train:        monthly returns 2000-2019 (for training)
      - ret_test:         monthly returns 2020-2023 (for OOS evaluation)
    """
    import yfinance as yf
    data   = yf.download('^GSPC', start=start, end=end, progress=False)
    closes = data['Close'].dropna().values.flatten()
    print(f"  S&P 500: {len(closes)} daily prices ({start[:4]}-{end[:4]})")
    log_ret_daily   = np.diff(np.log(closes))
    days_per_month  = 21
    n_months        = len(log_ret_daily) // days_per_month
    log_ret_monthly = np.array([
        log_ret_daily[i*days_per_month:(i+1)*days_per_month].sum()
        for i in range(n_months)
    ])
    # Split: first 239 months = 2000-2019 (training), rest = 2020-2023 (test)
    n_train   = 239
    ret_train = log_ret_monthly[:n_train]
    ret_test  = log_ret_monthly[n_train:]
    print(f"  Monthly returns: {n_train} train + {len(ret_test)} test")
    return log_ret_daily, ret_train, ret_test


# ─────────────────────────────────────────────────────────────────────────────
# 3.  CALIBRAZIONE
# ─────────────────────────────────────────────────────────────────────────────

def calibrate_bs(log_ret_monthly, r_f=R_F):
    sigma = np.std(log_ret_monthly) / np.sqrt(DT)
    mu    = np.mean(log_ret_monthly) / DT + 0.5 * sigma**2
    rho   = (mu - r_f) / sigma
    print(f"  BS:  mu={mu:.4f}  sigma={sigma:.4f}  rho={rho:.4f}")
    return dict(mu=mu, sigma=sigma, rho=rho, sJ2=0.)


def calibrate_mjd(log_ret_monthly, r_f=R_F):
    def neg_ll(params):
        mu, sigma, lam, m, delta = params
        if sigma <= 0 or lam <= 0 or delta <= 0:
            return 1e10
        kappa = np.exp(m + 0.5*delta**2) - 1
        n_max = max(10, int(lam*DT*5 + 20))
        ns    = np.arange(0, n_max+1)
        pn    = poisson.pmf(ns, lam*DT)
        pdf   = np.zeros(len(log_ret_monthly))
        for n, p_n in zip(ns, pn):
            if p_n < 1e-15: continue
            mu_n  = (mu - 0.5*sigma**2 - lam*kappa)*DT + n*m
            var_n = sigma**2*DT + n*delta**2
            pdf  += p_n * norm.pdf(log_ret_monthly, mu_n, np.sqrt(var_n))
        return -np.sum(np.log(np.maximum(pdf, 1e-300)))

    bs  = calibrate_bs(log_ret_monthly, r_f)
    x0  = [bs['mu'], bs['sigma'], 28.0, -0.004, 0.027]
    bds = [(None,None),(1e-4,None),(1e-4,None),(None,None),(1e-4,None)]
    res = minimize(neg_ll, x0, method='L-BFGS-B', bounds=bds,
                   options=dict(maxiter=500, ftol=1e-9))
    mu, sigma, lam, m, delta = res.x
    rho   = (mu - r_f) / sigma
    sJ2   = lam*(np.exp(2*m+2*delta**2) - 2*np.exp(m+0.5*delta**2) + 1)
    kappa = np.exp(m + 0.5*delta**2) - 1
    print(f"  MJD: mu={mu:.4f}  sigma={sigma:.4f}  lam={lam:.2f}"
          f"  m={m:.4f}  delta={delta:.4f}")
    print(f"       sigma_J^2={sJ2:.6f}")
    return dict(mu=mu, sigma=sigma, lam=lam, m=m, delta=delta,
                rho=rho, sJ2=sJ2, kappa=kappa)


def phi1_star(params, r_f=R_F):
    sigma = params['sigma']
    rho   = (params['mu'] - r_f) / sigma
    sJ2   = params.get('sJ2', 0.)
    return rho * sigma / (sigma**2 + sJ2)


# ─────────────────────────────────────────────────────────────────────────────
# 4.  SIMULATORI
# ─────────────────────────────────────────────────────────────────────────────

class BSSimulator:
    def __init__(self, p):
        self.sigma = p['sigma']
        self.rho   = (p['mu'] - R_F) / p['sigma']

    def step(self, x, a):
        dW = np.random.randn() * np.sqrt(DT)
        return x + a*self.sigma*self.rho*DT + a*self.sigma*dW


class MJDSimulator:
    def __init__(self, p):
        self.sigma = p['sigma']
        self.rho   = (p['mu'] - R_F) / p['sigma']
        self.lam   = p['lam']
        self.m     = p['m']
        self.delta = p['delta']
        self.kappa = p['kappa']

    def step(self, x, a):
        dW = np.random.randn() * np.sqrt(DT)
        dx = a*self.sigma*self.rho*DT + a*self.sigma*dW
        n  = np.random.poisson(self.lam * DT)
        if n > 0:
            Z   = np.random.normal(self.m, self.delta, n)
            dx += a * (float(np.sum(np.exp(Z) - 1)) - n*self.kappa)
        return x + dx


class RealDataSimulator:
    """
    Bootstrap resampling from historical monthly returns.
    Model-free approach consistent with Huang, Jia, Zhou (2022).
    """
    def __init__(self, log_ret_monthly):
        self.ret = log_ret_monthly

    def step(self, x, a):
        r = np.random.choice(self.ret)
        return x + a * (np.exp(r) - 1 - R_F * DT)


# ─────────────────────────────────────────────────────────────────────────────
# 5.  PARAMETRIZZAZIONI
# ─────────────────────────────────────────────────────────────────────────────

class MVModel:
    """
    Parameterizations (eq. 6.15-6.17 of the thesis):
      J^psi(t,x) = (x-omega)^2 * exp(-psi3*(T-t))
                 + psi2*(t^2-T^2) + psi1*(t-T) - (omega-z)^2
      q^phi(t,x,a) = -c(t)/2*(a + phi1*(x-omega))^2
                   - theta/2*[log(2*pi*theta) + phi2 + phi3*(T-t)]
      pi^phi(.|t,x) = N(-phi1*(x-omega), theta*c(t))
    """
    def __init__(self, psi3_init=0.):
        self.psi   = np.array([0., 0., psi3_init])
        self.phi   = np.zeros(3)
        self.omega = Z

    def _c(self, t):
        return np.exp(np.clip(-self.phi[1] - self.phi[2]*(T-t), -30, 30))

    def J(self, t, x):
        p = self.psi; o = self.omega
        e = np.exp(np.clip(-p[2]*(T-t), -30, 30))
        return (x-o)**2*e + p[1]*(t**2-T**2) + p[0]*(t-T) - (o-Z)**2

    def grad_J(self, t, x):
        p = self.psi; o = self.omega
        e = np.exp(np.clip(-p[2]*(T-t), -30, 30))
        return np.array([t-T, t**2-T**2, -(T-t)*(x-o)**2*e])

    def q(self, t, x, a):
        p = self.phi; o = self.omega; c = self._c(t)
        return (-c/2*(a + p[0]*(x-o))**2
                - THETA/2*(np.log(2*np.pi*THETA) + p[1] + p[2]*(T-t)))

    def grad_q(self, t, x, a):
        p = self.phi; o = self.omega; c = self._c(t)
        r = a + p[0]*(x-o)
        return np.array([c*r*(x-o),
                         0.5*c*r**2 - THETA/2,
                         (T-t)*0.5*c*r**2 - THETA*(T-t)/2])

    def sample_action(self, t, x):
        mu  = -self.phi[0] * (x - self.omega)
        var = max(THETA * self._c(t), 1e-8)
        return float(np.random.normal(mu, np.sqrt(var)))

    def mean_action(self, t, x):
        return -self.phi[0] * (x - self.omega)


# ─────────────────────────────────────────────────────────────────────────────
# 6.  ALGORITMO Q-LEARNING OFFLINE
# ─────────────────────────────────────────────────────────────────────────────

def offline_qlearning(simulator, label='', seed=0, psi3_init=0.):
    """
    Offline q-learning with Adam for phi and SGD with decay for psi.
    Adam eliminates manual calibration of alpha_phi
    (cf. Huang, Jia, Zhou 2022; Section B.5 of the thesis).
    """
    np.random.seed(seed)
    model = MVModel(psi3_init=psi3_init)
    hist  = np.zeros(N_ITER)

    m_phi = np.zeros(3); v_phi = np.zeros(3)
    adam_step = 0

    for j in range(1, N_ITER+1):
        lr    = j**(-DECAY)
        d_psi = np.zeros(3); d_phi = np.zeros(3); d_omega = 0.

        for _ in range(BATCH):
            xs = np.zeros(K+1); acts = np.zeros(K); xs[0] = X0
            for k in range(K):
                acts[k] = model.sample_action(k*DT, xs[k])
                nxt = simulator.step(xs[k], acts[k])
                xs[k+1] = nxt if np.isfinite(nxt) else xs[k]
            for k in range(K):
                td = (model.J((k+1)*DT, xs[k+1])
                      - model.J(k*DT, xs[k])
                      - model.q(k*DT, xs[k], acts[k]) * DT)
                if np.isfinite(td):
                    d_psi += model.grad_J(k*DT, xs[k]) * td / K
                    d_phi += model.grad_q(k*DT, xs[k], acts[k]) * td / K
            d_omega += xs[K] - Z

        # Gradient clipping
        for g in [d_psi, d_phi]:
            n = np.linalg.norm(g)
            if n > GRAD_CLIP * BATCH:
                g *= GRAD_CLIP * BATCH / n

        # SGD psi
        model.psi += lr * ALPHA_PSI * d_psi / BATCH

        # Adam phi
        adam_step += 1
        grad_phi = d_phi / BATCH
        m_phi    = ADAM_BETA1 * m_phi + (1 - ADAM_BETA1) * grad_phi
        v_phi    = ADAM_BETA2 * v_phi + (1 - ADAM_BETA2) * grad_phi**2
        m_hat    = m_phi / (1 - ADAM_BETA1**adam_step)
        v_hat    = v_phi / (1 - ADAM_BETA2**adam_step)
        model.phi += ADAM_LR * m_hat / (np.sqrt(v_hat) + ADAM_EPS)
        model.phi[1] = np.clip(model.phi[1], -10, 10)
        model.phi[2] = np.clip(model.phi[2], -10, 10)

        # SGD omega
        if j % OMEGA_FREQ == 0:
            model.omega -= lr * ALPHA_OMEGA * d_omega / (BATCH * OMEGA_FREQ)

        hist[j-1] = model.phi[0]

        if j % 2000 == 0:
            print(f"  [{label}] iter {j:5d}/{N_ITER}  "
                  f"phi_1={model.phi[0]:.4f}  omega={model.omega:.4f}")

    return hist, model


# ─────────────────────────────────────────────────────────────────────────────
# 7.  VALUTAZIONE OUT-OF-SAMPLE
# ─────────────────────────────────────────────────────────────────────────────

def terminal_wealth(model, simulator, n_paths=2000):
    """
    Evaluate the deterministic policy a_t = -phi1*(x_t - omega)
    on n_paths trajectories of the simulator.
    """
    tw = np.zeros(n_paths)
    for i in range(n_paths):
        x = X0
        for k in range(K):
            a = model.mean_action(k*DT, x)
            x = simulator.step(x, a)
        tw[i] = x
    return tw


# ─────────────────────────────────────────────────────────────────────────────
# 8.  MAIN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':

    print("=" * 60)
    print("  MV Portfolio — Offline q-Learning (Chapter 6, thesis)")
    print("=" * 60)

    print("\n[1] Loading S&P 500 data (2000-2023)...")
    log_ret_daily, log_ret_monthly, ret_test_chron = load_sp500()

    print("\n[2] Calibrating parameters (on 2000-2019 training data)...")
    bs_params  = calibrate_bs(log_ret_monthly)
    mjd_params = calibrate_mjd(log_ret_monthly)
    PHI1_BS    = phi1_star(bs_params)
    PHI1_MJD   = phi1_star(mjd_params)
    print(f"\n  phi_1* BS  = {PHI1_BS:.4f}")
    print(f"  phi_1* MJD = {PHI1_MJD:.4f}")

    psi3_init = ((bs_params['mu'] - R_F) / bs_params['sigma'])**2

    print("\n[3] Training on three simulators (Adam lr=0.001)...")
    real_sim = RealDataSimulator(log_ret_monthly)
    bs_sim   = BSSimulator(bs_params)
    mjd_sim  = MJDSimulator(mjd_params)

    print("\n  (A) Real S&P 500 data:")
    h_real, m_real = offline_qlearning(real_sim, 'Real', seed=1,
                                        psi3_init=psi3_init)
    print("\n  (B) BS simulator:")
    h_bs,   m_bs   = offline_qlearning(bs_sim,   'BS',   seed=2,
                                        psi3_init=psi3_init)
    print("\n  (C) MJD simulator:")
    h_mjd,  m_mjd  = offline_qlearning(mjd_sim,  'MJD',  seed=3,
                                        psi3_init=psi3_init)

    print(f"\n  Final phi_1:")
    print(f"    Real = {m_real.phi[0]:.4f}  "
          f"(benchmark BS={PHI1_BS:.4f}, MJD={PHI1_MJD:.4f})")
    print(f"    BS   = {m_bs.phi[0]:.4f}   (target {PHI1_BS:.4f})")
    print(f"    MJD  = {m_mjd.phi[0]:.4f}   (target {PHI1_MJD:.4f})")

    print("\n[4] Bootstrap OOS evaluation (2000 paths on real simulator)...")
    tw_real = terminal_wealth(m_real, real_sim)
    tw_bs   = terminal_wealth(m_bs,   real_sim)
    tw_mjd  = terminal_wealth(m_mjd,  real_sim)
    for lbl, tw in [('Real->Real', tw_real),
                    ('BS->Real',   tw_bs),
                    ('MJD->Real',  tw_mjd)]:
        print(f"  {lbl:<14}  E[X_T]={tw.mean():.4f}  std={tw.std():.4f}")

    # ── Chronological OOS evaluation 2020-2023 ───────────────────────────────
    print("\n[4b] Chronological OOS evaluation 2020-2023...")
    ret_test = ret_test_chron
    print(f"  Test set: {len(ret_test)} months (2020-2023)")

    def oos_chronological(model, ret_test, label):
        """Apply mean policy in strict chronological order."""
        x = X0
        wealth = [x]
        port_ret = []   # monthly portfolio returns (not log-wealth)
        for k, r in enumerate(ret_test):
            t = (k % K) * DT
            a = model.mean_action(t, x)
            r_excess = np.exp(r) - 1 - R_F * DT   # excess return of asset
            dx = a * r_excess
            port_ret.append(dx / x)                # portfolio % return this month
            x = x + dx
            wealth.append(x)
        wealth   = np.array(wealth)
        port_ret = np.array(port_ret)
        # Annualised return and vol from monthly portfolio returns
        ann_ret = port_ret.mean() * 12
        ann_vol = port_ret.std(ddof=1) * np.sqrt(12)
        sharpe  = (ann_ret - R_F) / ann_vol if ann_vol > 1e-8 else np.nan
        peak  = np.maximum.accumulate(wealth)
        maxdd = float(np.min(wealth / peak) - 1)
        return dict(XT=float(wealth[-1]), ann_ret=ann_ret,
                    ann_vol=ann_vol, sharpe=sharpe, maxdd=maxdd,
                    wealth=wealth)

    # Buy-and-hold: fixed allocation a = z - x0 = 0.4
    def oos_buyhold(ret_test, a_fixed=Z-X0):
        x = X0; wealth = [x]; port_ret = []
        for r in ret_test:
            r_excess = np.exp(r) - 1 - R_F * DT
            dx = a_fixed * r_excess
            port_ret.append(dx / x)
            x = x + dx
            wealth.append(x)
        wealth   = np.array(wealth)
        port_ret = np.array(port_ret)
        ann_ret = port_ret.mean() * 12
        ann_vol = port_ret.std(ddof=1) * np.sqrt(12)
        sharpe  = (ann_ret - R_F) / ann_vol if ann_vol > 1e-8 else np.nan
        peak  = np.maximum.accumulate(wealth)
        maxdd = float(np.min(wealth / peak) - 1)
        return dict(XT=float(wealth[-1]), ann_ret=ann_ret,
                    ann_vol=ann_vol, sharpe=sharpe, maxdd=maxdd,
                    wealth=wealth)

    # Analytical benchmark with given phi1
    def oos_benchmark(phi1, omega, ret_test):
        class BenchModel:
            def mean_action(self, t, x):
                return -phi1 * (x - omega)
        return oos_chronological(BenchModel(), ret_test, '')

    # For OOS evaluation, use omega=Z=1.4 (the portfolio target)
    # The training omega drifts slightly from Z due to stochastic approximation
    m_real.omega = Z

    oos_rl  = oos_chronological(m_real, ret_test, 'RL')
    oos_bh  = oos_buyhold(ret_test)
    oos_bs  = oos_benchmark(PHI1_BS,  Z, ret_test)
    oos_mjd = oos_benchmark(PHI1_MJD, Z, ret_test)

    print("\n" + "="*68)
    print("  CHRONOLOGICAL OOS 2020-2023 (tab:mv_oos)")
    print("="*68)
    print(f"  {'Strategy':<35} {'X_T':>6} {'Ret%':>7} {'Vol%':>7}"
          f" {'Sharpe':>7} {'MaxDD':>8}")
    print("  " + "-"*65)
    for lbl, s in [
        (f'Policy RL (phi1={m_real.phi[0]:.3f})',  oos_rl),
        ('Buy-and-hold (a_t = 0.4)',                oos_bh),
        (f'Benchmark BS (phi1*={PHI1_BS:.3f})',     oos_bs),
        (f'Benchmark MJD (phi1*={PHI1_MJD:.3f})',   oos_mjd),
    ]:
        print(f"  {lbl:<35} {s['XT']:>6.3f} {s['ann_ret']*100:>7.2f}"
              f" {s['ann_vol']*100:>7.2f} {s['sharpe']:>7.3f}"
              f" {s['maxdd']*100:>7.2f}%")
    print("="*68)

    print("\n[5] Saving results to mv_results.npz...")
    np.savez('mv_results.npz',
             h_real=h_real,    h_bs=h_bs,   h_mjd=h_mjd,
             tw_real=tw_real,  tw_bs=tw_bs, tw_mjd=tw_mjd,
             phi_real=m_real.phi,
             phi_bs=m_bs.phi,
             phi_mjd=m_mjd.phi,
             omega_real=np.array([m_real.omega]),
             omega_bs=np.array([m_bs.omega]),
             omega_mjd=np.array([m_mjd.omega]),
             PHI1_BS=np.array([PHI1_BS]),
             PHI1_MJD=np.array([PHI1_MJD]),
             oos_wealth_rl=oos_rl['wealth'],
             oos_wealth_bh=oos_bh['wealth'],
             oos_wealth_bs=oos_bs['wealth'],
             oos_wealth_mjd=oos_mjd['wealth'])
    print("  Saved: mv_results.npz")
    print("\n  Run mv_plot.py next to generate figures.")
    print("\nDone.")

  MV Portfolio — Offline q-Learning (Chapter 6, thesis)

[1] Loading S&P 500 data (2000-2023)...
  S&P 500: 6037 daily prices (2000-2023)
  Monthly returns: 239 train + 48 test

[2] Calibrating parameters (on 2000-2019 training data)...
  BS:  mu=0.0527  sigma=0.1649  rho=0.1981
  BS:  mu=0.0527  sigma=0.1649  rho=0.1981
  MJD: mu=0.0518  sigma=0.0668  lam=10.12  m=-0.0183  delta=0.0421
       sigma_J^2=0.020360

  phi_1* BS  = 1.2017
  phi_1* MJD = 1.2810

[3] Training on three simulators (Adam lr=0.001)...

  (A) Real S&P 500 data:
  [Real] iter  2000/20000  phi_1=0.6376  omega=1.4159
  [Real] iter  4000/20000  phi_1=0.9141  omega=1.4225
  [Real] iter  6000/20000  phi_1=1.0387  omega=1.4276
  [Real] iter  8000/20000  phi_1=1.1174  omega=1.4318
  [Real] iter 10000/20000  phi_1=1.1562  omega=1.4355
  [Real] iter 12000/20000  phi_1=1.2166  omega=1.4388
  [Real] iter 14000/20000  phi_1=1.1910  omega=1.4419
  [Real] iter 16000/20000  phi_1=1.2019  omega=1.4447
  [Real] iter 18000/20000  p